# CMPE 401 Project 2: run every experiment on a Colab GPU

Runs the whole benchmark in this repo: the Transformer/FordA reproduction and diagnostics, the official LSTM/Jena reproduction, the naive baselines, and the LSTM benchmark (L0 plus 11 single-change variants, 12 configurations × 3 seeds, then two combined models).

1. **Runtime → Change runtime type → T4 GPU**.
2. **Runtime → Run all**. Keep this tab open. The whole thing takes about 2–3 h on a T4.
3. With `PERSIST_TO_DRIVE = True`, finished runs are saved to `MyDrive/cmpe401-p2/`. If Colab disconnects, run all again: finished runs are skipped, an interrupted Transformer run resumes from its last epoch, and the LSTM benchmark continues from the last finished (variant, seed).
4. The last cell zips `results/` for download. Copy it into the repo's `results/` folder and commit.

In [ ]:
PERSIST_TO_DRIVE = True    # keep results/artifacts/data on Google Drive so a disconnect loses nothing
RUN_TRANSFORMER = True     # T0 official, T1 page version, T2 no encoder, T3 fixed, probe
RUN_LSTM_OFFICIAL = True   # L0 reproduction of the official example (official split)
RUN_LSTM_BENCHMARK = True  # naive baselines + L0 and 11 variants x SEEDS on the leak-free protocol
SEEDS = "0 1 2"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"
import tensorflow as tf, keras
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
%cd /content
!test -d p2 || git clone -q https://github.com/EzraKrause04/CMPE-401---Instructor-Defined-Project-2.git p2
%cd /content/p2
!git pull -q
!git log --oneline -1

In [ ]:
import os, shutil
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    store = "/content/drive/MyDrive/cmpe401-p2"
    for d in ["results", "artifacts", "data"]:
        os.makedirs(f"{store}/{d}", exist_ok=True)
        if os.path.isdir(d) and not os.path.islink(d):  # move anything already in the clone onto Drive
            shutil.copytree(d, f"{store}/{d}", dirs_exist_ok=True)
            shutil.rmtree(d)
        if not os.path.exists(d):
            os.symlink(f"{store}/{d}", d)
    print("results/, artifacts/, data/ ->", store)

## Transformer / FordA (Task 1 reproduction + diagnostics)

- **T0_official**: the current keras-io example, unchanged apart from logging.
- **T1_page_version**: `GlobalAveragePooling1D(data_format="channels_first")`, as it was before keras-io #1733 (Jan 2024). This matches the 93,130-parameter run on the rendered keras.io page.
- **T2_no_encoder**: T1 with zero Transformer blocks. It tests whether the encoder contributes anything.
- **T3_fixed**: a Conv1D input projection to 64 channels plus a positional embedding, so LayerNorm and attention have more than one feature to work on.
- **probe**: numerical checks that each encoder block computes `x + constant`.

In [ ]:
from pathlib import Path

def run(cmd, done_marker=None):
    """Run a shell command unless its finished-run marker exists; output streams into this cell."""
    if done_marker and Path(done_marker).exists():
        print(f"skip (finished): {cmd}")
        return
    print(f"$ {cmd}", flush=True)
    get_ipython().system(cmd)
    if rc := get_ipython().user_ns.get("_exit_code", 0):
        print(f"!! exit code {rc}: {cmd}")

if RUN_TRANSFORMER:
    t = "results/transformer"
    run("python experiments/transformer_official.py --run-name T0_official", f"{t}/T0_official/summary.json")
    run("python experiments/transformer_official.py --run-name T1_page_version --pooling channels_first", f"{t}/T1_page_version/summary.json")
    run("python experiments/transformer_official.py --run-name T2_no_encoder --pooling channels_first --blocks 0", f"{t}/T2_no_encoder/summary.json")
    run("python experiments/transformer_fixed.py --run-name T3_fixed", f"{t}/T3_fixed/summary.json")
    run("python experiments/transformer_probe.py --batch-size 16", f"{t}/probe.json")  # small batch: Colab has ~12.7 GB RAM

## LSTM / Jena Climate

- **L0_official**: the official example on its own train/validation split (Task 1 reproduction).
- **naive baselines**: persistence, same-time-yesterday and ridge regression, scored on the same test windows as the LSTMs.
- **benchmark**: the leak-free train/val/test protocol. There are 11 single-change variants plus the L0 baseline (12 configurations), each with 3 seeds, then two combined models built from the best single changes. Model selection uses val; results are reported on test.

In [ ]:
if RUN_LSTM_OFFICIAL:
    run("python experiments/lstm_official.py", "results/lstm/L0_official/summary.json")
if RUN_LSTM_BENCHMARK:
    run("python experiments/naive_baselines.py", "results/lstm_benchmark/naive_baselines.json")
    run(f"python experiments/lstm_benchmark.py --variant all --seeds {SEEDS}")  # skips finished (variant, seed) pairs itself
    # Combined models: the three best single changes by *validation* MAE (V4b, V8, V7), then + V3a.
    c1 = '{"shuffle": true, "time_features": true, "epochs": 30, "reduce_lr": true}'
    c2 = '{"shuffle": true, "time_features": true, "epochs": 30, "reduce_lr": true, "past": 360}'
    run(f"python experiments/lstm_benchmark.py --override '{c1}' --name C1_shuffle_time_plateau --seeds {SEEDS}")
    run(f"python experiments/lstm_benchmark.py --override '{c2}' --name C2_C1_past360 --seeds {SEEDS}")
    run("python experiments/aggregate_lstm.py")

In [ ]:
from IPython.display import Image, Markdown, display
for p in ["results/transformer/T0_official/curves.png", "results/transformer/T1_page_version/curves.png",
          "results/lstm/L0_official/loss_curve.png", "results/lstm_benchmark/test_mae.png"]:
    if Path(p).exists():
        display(Markdown(f"**{p}**")); display(Image(p))
if Path("results/lstm_benchmark/summary.md").exists():
    display(Markdown(Path("results/lstm_benchmark/summary.md").read_text()))

In [ ]:
!cd /content/p2 && zip -qr /content/p2_results.zip results/ && ls -lh /content/p2_results.zip
from google.colab import files
files.download("/content/p2_results.zip")